In [ ]:
import csv
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pickle
#from selenium import webdriver
#from selenium.webdriver.chrome.service import Service
#from webdriver_manager.chrome import ChromeDriverManager
#driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
#driver.get(link)

In [ ]:
template = 'https://www.hotnigerianjobs.com/jobs/{}/'

period_list = ['today', '1day', '2days', '3days']

url = template.format(period_list[1])

def get_page_urls(url):
    "Generate a url from period"

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    pages = soup.find('ol', {'id':'nav'})
    page_urls = set() # Initialize an empty set
    if pages: # Check if pages is not None
        pages_a = pages.find_all('a')
        page_urls = {link.get('href') for link in pages_a}

    return page_urls

In [ ]:
#urls = get_page_urls(url)
url

'https://www.hotnigerianjobs.com/jobs/1day/'

In [ ]:
import requests
from bs4 import BeautifulSoup

urls = get_page_urls(url)
#new_pages = get_page_urls('https://www.hotnigerianjobs.com/jobs/2days/5/')
new_pages = get_page_urls('https://www.hotnigerianjobs.com/jobs/1day/5/')
for page in new_pages:
  urls.add(page)

In [ ]:
urls

{'https://www.hotnigerianjobs.com/jobs/1day/0/',
 'https://www.hotnigerianjobs.com/jobs/1day/1/',
 'https://www.hotnigerianjobs.com/jobs/1day/2/',
 'https://www.hotnigerianjobs.com/jobs/1day/3/',
 'https://www.hotnigerianjobs.com/jobs/1day/4/',
 'https://www.hotnigerianjobs.com/jobs/1day/5/',
 'https://www.hotnigerianjobs.com/jobs/1day/6/',
 'https://www.hotnigerianjobs.com/jobs/1day/7/'}

In [ ]:
page3 = 'https://www.hotnigerianjobs.com/jobs/1days/8/'
page4 = 'https://www.hotnigerianjobs.com/jobs/1days/9/'
page5 = 'https://www.hotnigerianjobs.com/jobs/1days/10/'
page6 = 'https://www.hotnigerianjobs.com/jobs/1days/11/'
page7 = 'https://www.hotnigerianjobs.com/jobs/1days/12/'
#page4 = 'https://www.hotnigerianjobs.com/jobs/1days//'
urls.add(page3)
urls.add(page4)
urls.add(page5)
urls.add(page6)
urls.add(page7)

In [ ]:
urls

{'https://www.hotnigerianjobs.com/jobs/1day/0/',
 'https://www.hotnigerianjobs.com/jobs/1day/1/',
 'https://www.hotnigerianjobs.com/jobs/1day/2/',
 'https://www.hotnigerianjobs.com/jobs/1day/3/',
 'https://www.hotnigerianjobs.com/jobs/1day/4/',
 'https://www.hotnigerianjobs.com/jobs/1day/5/',
 'https://www.hotnigerianjobs.com/jobs/1day/6/',
 'https://www.hotnigerianjobs.com/jobs/1day/7/',
 'https://www.hotnigerianjobs.com/jobs/1days/10/',
 'https://www.hotnigerianjobs.com/jobs/1days/11/',
 'https://www.hotnigerianjobs.com/jobs/1days/12/',
 'https://www.hotnigerianjobs.com/jobs/1days/8/',
 'https://www.hotnigerianjobs.com/jobs/1days/9/'}

urls1 = list(urls)
urls_1 = urls1[:5]
urls_2 = urls1[5:]

In [ ]:
#job_links = {urls}
import re
def get_job_links(urls):
    #urls = get_page_urls()
    job_links = {}

    for url in urls:
        response1 = requests.get(url)
        soup1 = BeautifulSoup(response1.text, 'html.parser')
        links = soup1.find_all('span', {'class': 'jobheader'})
        for link in links:
            # Use regex to check for links with multiple jobs using the keyword "Position"
            search_text = re.compile(r'(\d)+(\s)?(Position)(\w)?(\))?(\s)?', re.I)
            key_text = search_text.search(link.find_next('h1').text)

            if key_text:
                #print(key_text.group())
                response2 = requests.get(link.a.get('href'))
                soup2 = BeautifulSoup(response2.text, 'html.parser')
                text = "Click Here To View Details"
                view_details = soup1.find_all(lambda tag: tag.name == "a" and text in tag.text)
                for item in view_details:
                    job_links[item.find_next('h1').text] = item.a.get('href')
            else:
                job_links[link.find_next('h1').text] = link.a.get('href')

    return job_links

In [ ]:
#search_text = re.compile(r'(\d)+(\s)*(Position)(\w)*(\))?(\s)?', re.I)
#key_text = search_text.search("40 position )")
#key_text.group()

In [ ]:
#%%timeit
jobs = get_job_links(urls)

In [ ]:
len(jobs)

321

In [ ]:
timeout_seconds = 60 # Increased timeout value

#%%timeit
text = 'Click here to apply online'
#all_jobs = {}
titles =[]
links = []
details = []
email_links = {}
web_links = {}
for k,v in jobs.items():
        try:
            response3 = requests.get(v, timeout=timeout_seconds)
            soup3 = BeautifulSoup(response3.text, 'html.parser')
            mycase4 = soup3.find('div', {'class':'mycase4'}) #get the parent "div" of job details
            if mycase4:
                detail = mycase4.find_all('div')[1] # The second div has the job details
                detail = ' '.join(detail.get_text().split('\n')) # get the text of all the elements in the div and convert the texts into a single string
            else:
                detail = k
            new_link = soup3.find('a', string=text)
            #print(new_link)
            titles.append(k)
            details.append(detail)
            if new_link:
                web_links[k] = new_link.get('href')
                links.append(new_link.get('href'))
            else:
                email_links[k] = v
                links.append(v)
        except requests.exceptions.ReadTimeout:
            print(f"ReadTimeout: Skipping {k} - {v} due to timeout after {timeout_seconds} seconds")
            # Optionally, you can add k, v to a list of failed jobs here
        except requests.exceptions.ConnectionError as e:
            print(f"ConnectionError: Skipping {k} - {v} due to connection error: {e}")
            # Optionally, you can add k, v to a list of failed jobs here
#print(titles[:5])

#%%timeit
text = 'Click here to apply online'
web_jobs = {}
email_jobs = {}
for k,v in jobs.items():
        response3 = requests.get(v)
        soup3 = BeautifulSoup(response3.text, 'html.parser')
        new_link = soup3.find('a', text=text)
        #print(new_link)
        if new_link:
            web_jobs[k] = new_link.get('href')
        else:
            email_jobs[k] = v

In [ ]:
#len(web_links)

In [ ]:
#a = web_links
#a

In [ ]:
#len(email_links)

In [ ]:
#b =email_links
#b

In [ ]:
#i= 1
#for k,v in web_links.items():
 #   print("{}. {} - {}".format(i, k, v))
 #   if i % 20 == 0:
  #      print("")
#print("")
   # i+=1

In [ ]:
#i= 121
#for k,v in email_links.items():
#    print("{}. {} - {}".format(i, k, v))
 #   if i % 20 == 0:
  #      print("")
    #    print("")
   # i+=1

In [ ]:
from datetime import date, timedelta
today = date.today()
post_date = today - timedelta(days = 1)
post_date = [post_date] * len(titles)

In [ ]:
all_jobs ={
    "Date": post_date,
    "Title": titles,
    "Link": links,
    "Detail": details
}

In [ ]:
df = pd.DataFrame.from_dict(all_jobs)
display(df.head(20))

,Date,Title,Link,Detail
0,2026-09-14,Direct Sales Agent at Catilas Resources Limited,https://www.hotnigerianjobs.com/hotjobs/958265...,Catilas Resources Limited is an outsourcing an...
1,2026-09-14,Production Manager at Tabs Atelier - Simply H...,https://www.hotnigerianjobs.com/hotjobs/958369...,Simply Human Resources Management (SimplyHRM) ...
2,2026-09-14,Sales & Marketing Professional at OmeFreight L...,https://www.hotnigerianjobs.com/hotjobs/958368...,OmeFreight Logistics Limited delivers integrat...
3,2026-09-14,LPG Manager at Coisco Integrated Resources & P...,https://forms.gle/yD6oZfZ5pwxi1KSN9,COISCO Integrated Resources and Petroleum Limi...
4,2026-09-14,Cook at Premiere Urgence Internationale (PUI),https://forms.cloud.microsoft/Pages/ResponsePa...,Première Urgence Internationale (PUI) is a non...
5,2026-09-14,Consultant Cardiologist at Lily Hospitals Limited,https://www.hotnigerianjobs.com/hotjobs/958356...,"Lily Hospitals Limited, established since 1986..."
6,2026-09-14,Safety & Hygiene Manager at Candace Beauty So...,https://www.hotnigerianjobs.com/hotjobs/958355...,"Candace Beauty Solutions is a premium, women-o..."
7,2026-09-14,Real Estate Sales Associate at a Real Estate /...,https://bit.ly/TYBITXAPPLICATION,Tybtitx Services International Limited - Our c...
8,2026-09-14,"Coverage Head, North West at Stanbic IBTC Bank",https://jobs.smartrecruiters.com/StandardBankG...,Stanbic IBTC Bank is a leading African banking...
9,2026-09-14,Project Director at Collection Development Lim...,https://www.hotnigerianjobs.com/hotjobs/958339...,"Collection Development Limited - We develop, m..."


In [ ]:
import re

def clean_illegal_characters(value):
    if isinstance(value, str):
        # Remove all illegal characters
        return re.sub(r'[\x00-\x1F]', '', value)
    return value

# Apply the cleaning function to the entire dataframe
df = df.map(clean_illegal_characters)

In [ ]:
all_jobs ={
    "Date": post_date,
    "Title": titles,
    "Link": links,
    "Detail": details
}

In [ ]:
df =pd.DataFrame.from_dict(all_jobs)

df.head(20)

,Date,Title,Link,Detail
0,2026-09-14,Direct Sales Agent at Catilas Resources Limited,https://www.hotnigerianjobs.com/hotjobs/958265...,Catilas Resources Limited is an outsourcing an...
1,2026-09-14,Production Manager at Tabs Atelier - Simply H...,https://www.hotnigerianjobs.com/hotjobs/958369...,Simply Human Resources Management (SimplyHRM) ...
2,2026-09-14,Sales & Marketing Professional at OmeFreight L...,https://www.hotnigerianjobs.com/hotjobs/958368...,OmeFreight Logistics Limited delivers integrat...
3,2026-09-14,LPG Manager at Coisco Integrated Resources & P...,https://forms.gle/yD6oZfZ5pwxi1KSN9,COISCO Integrated Resources and Petroleum Limi...
4,2026-09-14,Cook at Premiere Urgence Internationale (PUI),https://forms.cloud.microsoft/Pages/ResponsePa...,Première Urgence Internationale (PUI) is a non...
5,2026-09-14,Consultant Cardiologist at Lily Hospitals Limited,https://www.hotnigerianjobs.com/hotjobs/958356...,"Lily Hospitals Limited, established since 1986..."
6,2026-09-14,Safety & Hygiene Manager at Candace Beauty So...,https://www.hotnigerianjobs.com/hotjobs/958355...,"Candace Beauty Solutions is a premium, women-o..."
7,2026-09-14,Real Estate Sales Associate at a Real Estate /...,https://bit.ly/TYBITXAPPLICATION,Tybtitx Services International Limited - Our c...
8,2026-09-14,"Coverage Head, North West at Stanbic IBTC Bank",https://jobs.smartrecruiters.com/StandardBankG...,Stanbic IBTC Bank is a leading African banking...
9,2026-09-14,Project Director at Collection Development Lim...,https://www.hotnigerianjobs.com/hotjobs/958339...,"Collection Development Limited - We develop, m..."


In [ ]:
#with open("pickles/06th_Jan_2025_jobs.pkl", "wb") as f:
 #   pickle.dump(df2, f)

In [ ]:
#with open("pickles/06th_Jan_2025_jobs.pkl", "rb") as f:
 #   df = pickle.load(f)
#all_jobs = df.to_dict()

In [ ]:
#all_jobs["post_date"] = all_jobs["post_date"].values()
#all_jobs["titles"] = all_jobs["titles"].values()
#all_jobs["links"] = all_jobs["links"].values()
#all_jobs["details"] = all_jobs["details"].values()

#title_link = dict(zip(all_jobs["titles"], all_jobs["links"]))

In [ ]:
#i= 1
#for k,v in title_link.items():
 #   print("{}. {} - {}".format(i, k, v))
  #  if i % 20 == 0:
   #     print("")
    #    print("")
    #i+=1

In [ ]:
#from openpyxl import load_workbook

# To stop IllegalCharacterError:
#df = df.applymap(
    #lambda x: x.encode('unicode_escape')
   # .decode('utf-8') if isinstance(x, str) else x
#)

#rows = df.values.tolist()
#workbook = load_workbook(filename="hot_jobs_today.xlsx")
#sheet = workbook["job_list"]
#for row in rows:
 #   sheet.append(row)
#workbook.save(filename="hot_jobs_today.xlsx")

In [ ]:
pip install openpyxl

In [ ]:
import re

def clean_illegal_characters(value):
    if isinstance(value, str):
        # Remove all illegal characters
        return re.sub(r'[\x00-\x1F]', '', value)
    return value

# Apply the cleaning function to the entire dataframe
df = df.applymap(clean_illegal_characters)


/tmp/ipykernel_1188/479746676.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_illegal_characters)


In [ ]:
df.head()

,Date,Title,Link,Detail
0,2026-09-14,Direct Sales Agent at Catilas Resources Limited,https://www.hotnigerianjobs.com/hotjobs/958265...,Catilas Resources Limited is an outsourcing an...
1,2026-09-14,Production Manager at Tabs Atelier - Simply H...,https://www.hotnigerianjobs.com/hotjobs/958369...,Simply Human Resources Management (SimplyHRM) ...
2,2026-09-14,Sales & Marketing Professional at OmeFreight L...,https://www.hotnigerianjobs.com/hotjobs/958368...,OmeFreight Logistics Limited delivers integrat...
3,2026-09-14,LPG Manager at Coisco Integrated Resources & P...,https://forms.gle/yD6oZfZ5pwxi1KSN9,COISCO Integrated Resources and Petroleum Limi...
4,2026-09-14,Cook at Premiere Urgence Internationale (PUI),https://forms.cloud.microsoft/Pages/ResponsePa...,Première Urgence Internationale (PUI) is a non...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## Splitting into columns

In [ ]:
import pandas as pd


# Changed Type
df["Date"] = pd.to_datetime(df["Date"])
# Changed Type1
df["Date"] = df["Date"].dt.date

# Removed Columns
df.drop(columns=["Link"], inplace=True)

# Duplicated Column
df["Title - Copy"] = df["Title"]

# Removed Other Columns
df = df[["Date", "Title", "Detail", "Title - Copy"]]

# Split Column by Delimiter
df[["Job Title", "Company Name"]] = df["Title - Copy"].str.split(" at ", expand=True)
df.drop(columns=["Title - Copy"], inplace=True)

# Renamed Columns
df.rename(columns={"Job Title": "Job Title", "Company Name": "Company Name"}, inplace=True)

# Inserted Text Before Delimiter
df["Introduction"] = df["Detail"].str.split(".", 1).str[0]

# Inserted Text After Delimiter
df["Text After Delimiter"] = df["Detail"].str.split(" Type:", 1).str[-1].str.strip()

# Extracted Text Before Delimiter
df["Text After Delimiter"] = df["Text After Delimiter"].str.split(" ", 1).str[0]

# Replaced Value
df["Text After Delimiter"] = df["Text After Delimiter"].str.replace("Full time", "Full-time")

# Renamed Columns2
df.rename(columns={"Text After Delimiter": "Job Type"}, inplace=True)

# Inserted Text Between Delimiters
df["Location"] = df["Detail"].str.extract(r'Location: (.*?)\r')

# Replaced Value2
df["Location"] = df["Location"].str.replace("\xa0", "")

# Extracted Text Before Delimiter1
df["Location"] = df["Location"].str.split(".", 1).str[0]

# Inserted Text Between Delimiters1
df["Application Deadline"] = df["Detail"].str.extract(r'Application Closing Date(.*?)\.')

# Replaced Value3
df["Application Deadline"] = df["Application Deadline"].str.replace("\r", "")

# Trimmed Text
df["Application Deadline"] = df["Application Deadline"].str.strip()

# Inserted Text Between Delimiters2
df["Salary"] = df["Detail"].str.extract(r'Salary(.*?)\.')

# Replaced Value4
df["Salary"] = df["Salary"].str.replace("\r", "")

# Trimmed Text
df["Salary"] = df["Salary"].str.strip()

# Renamed Columns6
df["Specialization"] = df["Detail"].str.split("Requirements", 1).str[-1].str.strip()

# Drop unnecessary columns
df.drop(columns=["Detail","Title"], inplace=True)

# Finally, you may want to reorder the columns if needed
# df = df[["Column1", "Date", "Title", "Job Title", "Company Name", "Introduction", "Job Type", "Location", "Application Deadline", "Salary", "Specialization"]]

# Print or use df as needed
df.head(10)


In [ ]:
# Export the dataframe to an Excel file
df.to_excel('Online_vacancy_September_14.xlsx', sheet_name='Sheet1')
# Assuming 'df' is your DataFrame
print('DataFrame is written successfully to Excel File.')

DataFrame is written successfully to Excel File.
